# Generamos datos sinteticos de fincas con 'Faker', Pandas y Numpy:

In [1]:
# Importaciones

import pandas as pd 
import numpy as np 
from faker import Faker 
import random

fake = Faker('es_AR')
Faker.seed(42)
np.random.seed(42)


In [2]:
# 1. Generacion de datos de fincas y lotes:

fincas = [f'Finca {i}' for i in range(1,6)]  # Generar nombres de fincas del 1 al 5']
lotes_data = []

for i in range(1,41): #40 lotes en total
    lotes_data.append({
    'lote_id': f'LOT-{i:03d}',  # Generar IDs de lotes del 001 al 040
    'nombre_finca': random.choice(fincas),  # Asignar una finca aleatoria a cada lote
    'superficie_ha':round(random.uniform(15, 500), 1),  # Generar superficie aleatoria entre 15 y 50 hectáreas
    'variedad': random.choice(['Limon Eureka', 'Limon Lisboa', 'Genova']),  # Asignar una variedad de limón aleatoria a cada lote
    'edad_monte_anos': random.randint(5, 25),  # Generar edad del monte aleatoria entre 5 y 25 años
    'rinde_historico_t_ha': round(np.random.normal(35, 4), 1) # Generar rinde histórico con media de 35 t/ha y desviación estándar de 4

    })
    
df_lotes = pd.DataFrame(lotes_data)

# Generar datos climaticos semanales:

fechas = pd.date_range(start='2010-01-01', end='2026-08-01', freq='W-MON') # Generar fechas semanales desde el 1 de enero de 2010 hasta el 1 de agosto de 2026

clima_list = []

for fecha in fechas:
    mes = fecha.month
    
    # Lluvias concentradas en verano
    if mes in [11, 12, 1, 2, 3]:  # Verano / Primavera tardía
        lluvia_mm = max(0, np.random.gamma(shape=3, scale=12))
        temp_media = np.random.normal(26, 3)
    elif mes in [4, 5, 9, 10]:    # Transición
        lluvia_mm = max(0, np.random.gamma(shape=1.5, scale=8))
        temp_media = np.random.normal(20, 3)
    else:                          # Invierno seco (Junio, Julio, Agosto)
        lluvia_mm = max(0, np.random.gamma(shape=0.5, scale=4))
        temp_media = np.random.normal(14, 4)
        
    clima_list.append({
        'fecha': fecha.strftime('%Y-%m-%d'),
        'mes': mes,
        'anio': fecha.year,
        'lluvia_semanal_mm': round(lluvia_mm, 1),
        'temperatura_media_c': round(temp_media, 1),
        'riesgo_helada': 1 if (mes in [6, 7] and temp_media < 5) else 0
    })

df_clima = pd.DataFrame(clima_list)

# Generar datos de yield

rindes_anuales = []

anios = df_clima['anio'].unique()

for anio in anios:
    # 1. Muestrear el clima acumulado de ese año
    clima_anio = df_clima[df_clima['anio'] == anio]
    lluvia_total_mm = clima_anio['lluvia_semanal_mm'].sum()
    heladas_totales = clima_anio['riesgo_helada'].sum()
    
    # 2. Calcular el factor de impacto climático (1.0 = normal)
    factor_clima = 1.0
    
    # Penalización por Sequía (Si llueve menos de 700 mm en el año)
    if lluvia_total_mm < 700:
        factor_clima -= 0.25  # Pierde 25% de rinde
    elif lluvia_total_mm < 900:
        factor_clima -= 0.10  # Pierde 10% de rinde
        
    # Penalización por Heladas (Severidad según cantidad de semanas con helada)
    if heladas_totales >= 3:
        factor_clima -= 0.30  # Helada fuerte: pierde 30%
    elif heladas_totales >= 1:
        factor_clima -= 0.15  # Helada moderada: pierde 15%
        
    # Evitar factores negativos o absurdos
    factor_clima = max(0.4, factor_clima)
    
    # 3. Asignar el rinde real ajustado a cada lote para esa campaña
    for _, lote in df_lotes.iterrows():
        # Variabilidad aleatoria propia del lote + efecto climático
        ruido_lote = np.random.normal(0, 1.5)
        rinde_real = (lote['rinde_historico_t_ha'] * factor_clima) + ruido_lote
        
        rindes_anuales.append({
            'anio': anio,
            'lote_id': lote['lote_id'],
            'nombre_finca': lote['nombre_finca'],
            'lluvia_acumulada_mm': round(lluvia_total_mm, 1),
            'heladas_registradas': heladas_totales,
            'factor_clima': round(factor_clima, 2),
            'rinde_real_t_ha': round(max(10.0, rinde_real), 1),
            'produccion_total_t': round(max(10.0, rinde_real) * lote['superficie_ha'], 1)
        })

df_rindes = pd.DataFrame(rindes_anuales)

# Generar datos de mercado (precio de insumos y del limon):

mercado_list = []

for i, fecha in enumerate(fechas):
    # Tendencia de incremento de costos a lo largo del tiempo 
    factor_tiempo = 1 + (i / len(fechas)) * 0.15 
    
    # Precios de Insumos (USD)
    p_urea = (0.55 + np.random.normal(0, 0.04)) * factor_tiempo
    p_fungicida = (16.50 + np.random.normal(0, 1.2)) * factor_tiempo
    p_gasoil = (1.05 + np.random.normal(0, 0.05)) * factor_tiempo
    
    # Precios Limón (USD/t) con volatilidad de mercado
    p_exportacion = 450 + np.random.normal(0, 40) + (np.sin(i / 10) * 30)
    p_industria = 60 + np.random.normal(0, 12)
    
    mercado_list.append({
        'fecha': fecha.strftime('%Y-%m-%d'),
        'precio_urea_usd_kg': round(max(0.30, p_urea), 2),
        'precio_fungicida_usd_l': round(max(10.0, p_fungicida), 2),
        'precio_gasoil_usd_l': round(max(0.70, p_gasoil), 2),
        'precio_limon_exportacion_usd_t': round(p_exportacion, 2),
        'precio_limon_industria_usd_t': round(max(20.0, p_industria), 2)
    })

df_mercado = pd.DataFrame(mercado_list)

# Generar datos de demanda de insumos y costos

registros_operativos = []

for _, lote in df_lotes.iterrows():
    for _, clima in df_clima.iterrows():
        fecha_str = clima['fecha']
        mes = clima['mes']
        mkt = df_mercado[df_mercado['fecha'] == fecha_str].iloc[0]
        
        # Fertilizante (de Septiembre a Enero)
        kg_urea_ha = 0
        if mes in [9, 10, 11, 12, 1]:
            # Proporcional al rinde historico del lote
            dosis_base = (lote['rinde_historico_t_ha'] / 35) * 40  # kg/ha por aplicación
            kg_urea_ha = max(0, np.random.normal(dosis_base, 5))
            
        # Fungicida
        litros_fungicida_ha = 0
        # Septiembre a Diciembre
        if mes in [9, 10, 11, 12]:
            if clima['lluvia_semanal_mm'] > 35 or random.random() < 0.25:
                litros_fungicida_ha = np.random.normal(2.2, 0.3)
                
        # Labores y Combustible
        # Aumenta en cosecha (Abril-Agosto) y en pulverizaciones
        gasoil_base = 6.0 if mes in [4, 5, 6, 7, 8] else 3.0
        if litros_fungicida_ha > 0: 
            gasoil_base += 2.5  # Pasada de tractor
        litros_gasoil_ha = np.random.normal(gasoil_base, 0.8)
        
        # Cálculo de Costos Fincados
        costo_urea = kg_urea_ha * mkt['precio_urea_usd_kg']
        costo_fungicida = litros_fungicida_ha * mkt['precio_fungicida_usd_l']
        costo_gasoil = litros_gasoil_ha * mkt['precio_gasoil_usd_l']
        costo_total_ha = costo_urea + costo_fungicida + costo_gasoil
        
        registros_operativos.append({
            'fecha': fecha_str,
            'lote_id': lote['lote_id'],
            'nombre_finca': lote['nombre_finca'],
            'demanda_urea_kg_ha': round(kg_urea_ha, 1),
            'demanda_fungicida_l_ha': round(litros_fungicida_ha, 1),
            'demanda_gasoil_l_ha': round(litros_gasoil_ha, 1),
            'costo_insumos_usd_ha': round(costo_total_ha, 2),
            'costo_total_lote_usd': round(costo_total_ha * lote['superficie_ha'], 2)
        })

df_operativo = pd.DataFrame(registros_operativos)

# Verificacion de la integridad de los datos generados

print(f"Total de Lotes generados: {len(df_lotes)}")
print(f"Semanas simuladas (2010-2026): {len(df_clima)}")
print(f"Registros operativos totales (Lotes x Semanas): {len(df_operativo):,}")

print("\n--- MUESTRA DE DATOS OPERATIVOS ---")
print(df_operativo[df_operativo['demanda_fungicida_l_ha'] > 0].head())



Total de Lotes generados: 40
Semanas simuladas (2010-2026): 865
Registros operativos totales (Lotes x Semanas): 34,600

--- MUESTRA DE DATOS OPERATIVOS ---
         fecha  lote_id nombre_finca  demanda_urea_kg_ha  \
35  2010-09-06  LOT-001      Finca 3                52.9   
36  2010-09-13  LOT-001      Finca 3                50.9   
39  2010-10-04  LOT-001      Finca 3                38.4   
43  2010-11-01  LOT-001      Finca 3                51.2   
44  2010-11-08  LOT-001      Finca 3                45.6   

    demanda_fungicida_l_ha  demanda_gasoil_l_ha  costo_insumos_usd_ha  \
35                     1.4                  5.4                 60.87   
36                     2.2                  6.1                 68.72   
39                     2.0                  5.1                 59.73   
43                     2.1                  3.6                 68.39   
44                     2.5                  4.2                 69.70   

    costo_total_lote_usd  
35              2

In [3]:
# Guardamos datos generados como archivos CSV

df_lotes.to_csv('dim_lotes.csv', index=False)
df_operativo.to_csv('hecho_historico_operativo.csv', index=False)
df_clima.to_csv('dim_clima.csv', index=False)
df_mercado.to_csv('dim_mercado.csv', index=False)